# Fall Detection Training (Random Forest)
This notebook trains a Random Forest classifier to distinguish between Activities of Daily Living (ADL) and actual falls using the SisFall dataset.

### Cell 1: Setup
We import standard machine learning tools and our custom data ingestion logic.

In [ ]:
%run ../data/Data_Management.ipynb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import joblib, os, numpy as np
from tqdm.auto import tqdm
loader = DataLoader()

### Cell 2: Multi-Subject Ingestion
We iterate through subjects SA01-SE01 and multiple trials of ADLs (D01-D04) and Falls (F01-F04). Each file is cleaned using `clean_sisfall_data`.

In [ ]:
subjects, trials = ["SA01", "SA02", "SA03", "SE01"], ["R01", "R02"]
activities = [f"D{i:02d}" for i in range(1, 5)] + [f"F{i:02d}" for i in range(1, 5)]
X_list, y_list = [], []
for sub in tqdm(subjects, desc="SisFall"):
    for act in activities:
        for trial in trials:
            fname = f"{act}_{sub}_{trial}.txt"
            df = loader.load_sisfall_file(sub, fname)
            if df is not None:
                df_c = clean_sisfall_data(df)
                X_list.append(df_c.values)
                y_list.append(np.full(len(df_c), 1 if act.startswith("F") else 0))
X_all, y_all = np.vstack(X_list), np.concatenate(y_list)
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

### Cell 3: Random Forest Training
The Random Forest is trained with 100 trees. We use `n_jobs=-1` to utilize all CPU cores for faster training.

In [ ]:
fall_model = RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1)
fall_model.fit(X_train, y_train)
print(classification_report(y_test, fall_model.predict(X_test)))
joblib.dump(fall_model, "../../models/fall_detection_v1.joblib")